In [1]:
import pandas as pd
import ast
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import re


In [6]:
google_df = pd.read_csv('data/google.csv')
uk = pd.read_csv('data/uk.csv')
aus = pd.read_csv('data/australia.csv')
ind = pd.read_csv('data/india.csv')

combined = pd.concat([google_df, uk, ind, aus], ignore_index=True)

combined["query"] = combined['query'].str.lower()

In [2]:
google_df = pd.read_csv('data/google.csv')
baidu_df = pd.read_csv('data/baidu.csv')

# Normalize query column name
baidu_df["query"] = baidu_df["query_en"]

google_df['query'] = google_df['query'].str.lower()
baidu_df["query_en"] = baidu_df["query_en"].str.lower()

# Add platform columns
google_df["platform"] = "google"
baidu_df["platform"] = "baidu"
baidu_df["location"] = "china"


In [3]:
google_df

,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform
0,0,United States,breast cancer,True,['A new lump or mass in the breast or underarm...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",google
1,1,United States,what is cancer,True,['Tumor Formation: Extra cells form masses of ...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",google
2,2,United States,cancer symptoms,True,['Unexplained Weight Loss: Losing $\\ge 10$ po...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",google
3,3,United States,lung cancer,True,['Non-Small Cell Lung Cancer (NSCLC): Accounts...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,google
4,4,United States,skin cancer,True,"[""Basal Cell Carcinoma (BCC): The most common ...","[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,google
...,...,...,...,...,...,...,...,...,...
113,113,United States,chinese herbal medicine,True,[],[],[],[],google
114,114,United States,supreme court abortion,True,"['The Dobbs Decision: In 2022, the Court ruled...",[{'title': 'Roe v. Wade and Supreme Court Abor...,[{'title': 'Roe v. Wade and Supreme Court Abor...,['What is the new Supreme Court decision on ab...,google
115,115,United States,hiv treatment,True,['Undetectable = Untransmittable (U=U): When y...,"[{'title': 'Treating HIV - CDC', 'link': 'http...","[{'title': 'Treating HIV - CDC', 'link': 'http...","['Can HIV be treated completely?', 'What are 7...",google
116,116,United States,cupping near me,False,NaN,NaN,NaN,"['Does cupping reduce lipedema?', 'Can you get...",google


In [5]:
TOPIC_MAP = {
    # --- Cancer ---
    "breast cancer": "cancer",
    "what is cancer": "cancer",
    "cancer symptoms": "cancer",
    "lung cancer": "cancer",
    "skin cancer": "cancer",
    "colon cancer": "cancer",
    "cancer treatment": "cancer",
    "prostate cancer": "cancer",
    "blood cancer": "cancer",
    "pancreatic cancer": "cancer",
    # --- Diabetes ---
    "diabetes type 2": "diabetes",
    "type 2 diabetes": "diabetes",
    "what is diabetes": "diabetes",
    "diabetes type 1": "diabetes",
    "type 1 diabetes": "diabetes",
    "diabetes symptoms": "diabetes",
    "gestational diabetes": "diabetes",
    "insulin": "diabetes",
    "symptoms of diabetes": "diabetes",
    "diabetes test": "diabetes",
    # --- Hypertension ---
    "hypertension blood pressure": "hypertension",
    "blood pressure": "hypertension",
    "pulmonary hypertension": "hypertension",
    "what is hypertension": "hypertension",
    "hypertension symptoms": "hypertension",
    "high blood pressure": "hypertension",
    "hypertension treatment": "hypertension",
    "intracranial hypertension": "hypertension",
    "stage 2 hypertension": "hypertension",
    "hypertension stage 2": "hypertension",
    # --- Depression ---
    "anxiety": "depression",
    "what is depression": "depression",
    "anxiety and depression": "depression",
    "depression and anxiety": "depression",
    "depression treatment": "depression",
    "depression symptoms": "depression",
    "depression help": "depression",
    "postpartum": "depression",
    "postpartum depression": "depression",
    "depression medication": "depression",
    # --- Hantavirus ---
    "the hantavirus": "hantavirus",
    "virus": "hantavirus",
    "hantavirus symptoms": "hantavirus",
    "virus hantavirus": "hantavirus",
    "hantavirus outbreak": "hantavirus",
    "virus hantavirus outbreak": "hantavirus",
    "what is hantavirus": "hantavirus",
    "hantavirus 2026": "hantavirus",
    "hantavirus cruise": "hantavirus",
    "hantavirus update": "hantavirus",
    # --- Long COVID ---
    "covid symptoms": "long_covid",
    "long covid symptoms": "long_covid",
    "what is long covid": "long_covid",
    "how long does covid last": "long_covid",
    "long term covid": "long_covid",
    "how long was covid": "long_covid",
    "symptoms of covid": "long_covid",
    "symptoms of long covid": "long_covid",
    "long covid treatment": "long_covid",
    "covid 19": "long_covid",
    # --- Acupuncture ---
    "acupuncture near me": "acupuncture",
    "what is acupuncture": "acupuncture",
    "acupuncture points": "acupuncture",
    "acupuncture for pain": "acupuncture",
    "chinese acupuncture": "acupuncture",
    "community acupuncture": "acupuncture",
    "does acupuncture work": "acupuncture",
    "fertility acupuncture": "acupuncture",
    "acupuncture needles": "acupuncture",
    "acupuncture": "acupuncture",
    # --- Cupping ---
    "cupping therapy near me": "cupping",
    "cupping near me": "cupping",
    "what is cupping therapy": "cupping",
    "cupping massage therapy": "cupping",
    "massage cupping therapy": "cupping",
    "cupping massage": "cupping",
    "what is cupping": "cupping",
    "cupping therapy benefits": "cupping",
    "cupping set": "cupping",
    "massage": "cupping",

    # --- Herbal medicine ---
    "herbal chinese medicine": "herbal",
    "chinese medicine": "herbal",
    "chinese herbal medicine": "herbal",
    "what is herbal medicine": "herbal",
    "traditional herbal medicine": "herbal",
    "herbal medicine near me": "herbal",
    "acupuncture herbal medicine": "herbal",
    "herbal medicine school": "herbal",
    "herbal tea": "herbal",
    # --- Abortion ---
    "what is abortion": "abortion",
    "abortion clinic": "abortion",
    "abortion laws": "abortion",
    "planned parenthood": "abortion",
    "planned parenthood abortion": "abortion",
    "medical abortion": "abortion",
    "what is an abortion": "abortion",
    "supreme court abortion": "abortion",
    "abortion pill online": "abortion",
    "abortion meaning": "abortion",
    # --- HIV ---
    "the hiv": "hiv",
    "what hiv": "hiv",
    "what is hiv": "hiv",
    "aids": "hiv",
    "hiv aids": "hiv",
    "hiv symptoms": "hiv",
    "hiv test": "hiv",
    "hiv positive": "hiv",
    "hiv treatment": "hiv",
    "hiv testing": "hiv",
    # --- Gender-affirming care ---
    "gender affirming care": "gender_affirming",
    "gender affirming surgery": "gender_affirming",
    "what is gender affirming care": "gender_affirming",
    "gender affirming care definition": "gender_affirming",
    "gender affirming care near me": "gender_affirming",
    "gender reassignment surgery": "gender_affirming",
    "gender reassignment surgery male to female": "gender_affirming",
    "how does gender reassignment surgery work": "gender_affirming",
    "gender affirming hormone therapy": "gender_affirming",
    "gender affirmation surgery": "gender_affirming"
}

SECTION_MAP = {
    "cancer": "common_conditions",
    "diabetes": "common_conditions",
    "hypertension": "common_conditions",
    "depression": "controversial",
    "hantavirus": "controversial",
    "long_covid": "controversial",
    "acupuncture": "tcm",
    "cupping": "tcm",
    "herbal": "tcm",
    "abortion": "legally_variable",
    "hiv": "legally_variable",
    "gender_affirming": "legally_variable" 
}

for df in [baidu_df, google_df]:
    df["topic"]   = df["query"].map(TOPIC_MAP)
    df["section"] = df["topic"].map(SECTION_MAP)

# Sanity check
# for name, df in [("Google", google_df), ("Baidu", baidu_df)]:
#     unmapped = df[df["topic"].isna()]["query"].tolist()
#     if unmapped:
#         print(f"{name} — unmapped queries: {unmapped}")
#     else:
#         print(f"{name} — all queries mapped ✓")

In [13]:
test = pd.concat([google_df, uk, ind, aus], ignore_index=True)
test['has_ai_overview'].mean()

np.float64(0.8200836820083682)

In [7]:
combined = pd.concat([google_df, baidu_df], ignore_index=True)

combined['has_ai_overview'] = combined['all_items'].apply(lambda x: x not in [None, "[]"] and not (isinstance(x, float) and pd.isna(x)))


In [8]:
combined

,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform,topic,section,query_en,query_zh,related_searches
0,0.0,United States,breast cancer,True,['A new lump or mass in the breast or underarm...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",google,cancer,common_conditions,NaN,NaN,NaN
1,1.0,United States,what is cancer,True,['Tumor Formation: Extra cells form masses of ...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN
2,2.0,United States,cancer symptoms,True,['Unexplained Weight Loss: Losing $\\ge 10$ po...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",google,cancer,common_conditions,NaN,NaN,NaN
3,3.0,United States,lung cancer,True,['Non-Small Cell Lung Cancer (NSCLC): Accounts...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,google,cancer,common_conditions,NaN,NaN,NaN
4,4.0,United States,skin cancer,True,"[""Basal Cell Carcinoma (BCC): The most common ...","[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,google,cancer,common_conditions,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,NaN,china,gender affirming surgery,True,"['\u200c术前评估\u200c', '\u200c手术类型\u200c', '\u20...",[],[],NaN,baidu,gender_affirming,legally_variable,gender affirming surgery,性别重置手术,"['变性一年后外阴效果图', '去势手术', '男生被绑在手术台上做变性手术二次', '40..."
234,NaN,china,gender reassignment surgery,False,NaN,NaN,NaN,NaN,baidu,gender_affirming,legally_variable,gender reassignment surgery,性别转换手术,"['性别转换生成器', '一觉醒来我就变成了女孩子', '男变女手术', '变性一年后外阴效..."
235,NaN,china,gender reassignment surgery male to female,True,['\u200c心理评估与法律程序\u200c 需接受至少12个月的心理评估，由精神科医生确...,[],[],NaN,baidu,gender_affirming,legally_variable,gender reassignment surgery male to female,男变女手术,"['男性割掉下体手术', '男科手术视频(真人)', '女性外阴切除手术实拍', '变性一年..."
236,NaN,china,how does gender reassignment surgery work,False,NaN,NaN,NaN,NaN,baidu,gender_affirming,legally_variable,how does gender reassignment surgery work,变性手术如何进行,"['男变性手术', '做变性手术需要多少钱', '女的变男的变性手术是怎么变', '怎样变性..."


In [9]:
combined['has_ai_overview'].mean()

np.float64(0.6386554621848739)

In [10]:
combined.to_csv("df.csv")